In [0]:
# Databricks notebook source
# ============================================================
# 🔶 BRONZE INGESTION — SOURCE 3: SharePoint/Excel (L&D Tracker)
# ============================================================
# Tables: certification (upsert), skill_readiness (upsert),
#         training_feedback (append)
# ============================================================

In [0]:
%run ../05_utils/common_functions

In [0]:
# CELL 3: Define SharePoint config
sp_config = [
    {
        "source_system": "SharePoint",
        "source_path": f"/Volumes/{CATALOG}/{SCHemaEMA}/sharepoint_files/certification.csv",
        "target_table": f"{CATALOG}.{SCHEMA}.sp_certification",
        "load_type": "upsert",       # certificate_status changes (Pending→Verified)
        "primary_key": "certification_id"
    },
    {
        "source_system": "SharePoint",
        "source_path": f"/Volumes/{CATALOG}/{SCHEMA}/sharepoint_files/skill_workforce_readiness.csv",
        "target_table": f"{CATALOG}.{SCHEMA}.sp_skill_readiness",
        "load_type": "upsert",       # skills_verified, skills_declared change
        "primary_key": "skill_id"
    },
    {
        "source_system": "SharePoint",
        "source_path": f"/Volumes/{CATALOG}/{SCHEMA}/sharepoint_files/training_feedback.csv",
        "target_table": f"{CATALOG}.{SCHEMA}.sp_training_feedback",
        "load_type": "append",       # feedback once given doesn't change
        "primary_key": "feedback_id"
    }
]

print(f"📋 SharePoint Config: {len(sp_config)} tables")
for c in sp_config:
    print(f"   {c['target_table'].split('.')[-1]:<35} → {c['load_type']}")

📋 SharePoint Config: 3 tables
   sp_certification                    → upsert
   sp_skill_readiness                  → upsert
   sp_training_feedback                → append


In [0]:
# CELL 4: Execute
from datetime import datetime

start = datetime.now()
print(f"🚀 SHAREPOINT INGESTION STARTED: {start.strftime('%H:%M:%S')}")

sp_results = ingest_to_bronze(sp_config)

end = datetime.now()
print(f"\n{'═'*65}")
print(f"📊 SHAREPOINT INGESTION COMPLETE — {(end-start).total_seconds():.1f}s")
print(f"{'═'*65}")
for r in sp_results:
    print(f"   {r['status']} {r['table']:<35} {r['mode']:<7} → {r['rows']} rows")

🚀 SHAREPOINT INGESTION STARTED: 12:44:56

─────────────────────────────────────────────────────────────────
⏳ [SharePoint] sp_certification
   Source : /Volumes/hackathon_ltm/bronze/sharepoint_files/certification_table.csv
   Target : hackathon_ltm.bronze.sp_certification
   Mode   : UPSERT | Key: certification_id
─────────────────────────────────────────────────────────────────
   📄 Records read: 500
   📝 Table doesn't exist → CREATING
   ✅ CREATED: 500 rows

─────────────────────────────────────────────────────────────────
⏳ [SharePoint] sp_skill_readiness
   Source : /Volumes/hackathon_ltm/bronze/sharepoint_files/skill_workforce_readiness.csv
   Target : hackathon_ltm.bronze.sp_skill_readiness
   Mode   : UPSERT | Key: skill_id
─────────────────────────────────────────────────────────────────
   📄 Records read: 500
   📝 Table doesn't exist → CREATING
   ✅ CREATED: 500 rows

─────────────────────────────────────────────────────────────────
⏳ [SharePoint] sp_training_feedback
   Sourc